## Hypothesis Testing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind

df = pd.read_csv('/content/drive/MyDrive/ImpactSense_Oct25/data/preprocessed_earthquake_data.csv')
df.head()

,latitude,longitude,depth,mag,magType,rms,type,status,Year,Month,Day,Hour,Minute,Second,DayOfWeek
0,41.758,23.249,-0.430517,3.296720,mw,0.255959,earthquake,reviewed,1904,4,4,10,26,0,0
1,41.802,23.108,-0.430517,2.920854,mw,0.227615,earthquake,reviewed,1904,4,4,10,2,34,0
2,52.763,160.277,-0.291162,4.716657,mw,0.360656,earthquake,reviewed,1904,6,25,21,0,38,5
3,51.424,161.638,-0.430517,4.299028,mw,0.331543,earthquake,reviewed,1904,6,25,14,45,39,5
4,30.684,100.608,-0.430517,3.442890,mw,0.266982,earthquake,reviewed,1904,8,30,11,43,20,1


#### 1. Magnitude Difference by Earthquake Type
Hypothesis: The mean earthquake magnitude differs significantly between reviewed and non-reviewed events.

In [ ]:
# Select relevant groups, drop missing values
group1 = df[df['status'] ==  'reviewed']['mag'].dropna()
group2 = df[df['status'] !=  'reviewed']['mag'].dropna()

ttest_result = ttest_ind(group1, group2, equal_var=False)
print('t-test statistics:', ttest_result.statistic)
print('p-value:', ttest_result.pvalue)

t-test statistics: -1.0748565111189015
p-value: 0.28494966597262317


**Statement**:The t-test result for Hypothesis-1 yields a t-statistic of -1.07 and a p-value of 0.285. Since the p-value is greater than the common significance threshold of 0.05, there is not enough evidence to reject the null hypothesis. Thus, the mean earthquake magnitude does not differ significantly between reviewed and non-reviewed events in your data.

#### 2. Depth Difference by Magnitude Type
Hypothesis: The mean depth of earthquakes significantly varies between 'mb' and 'mw' magnitude type categories.

In [ ]:
group1 = df[df['magType'] == 'mb']['depth'].dropna()
group2 = df[df['magType'] == 'mw']['depth'].dropna()

ttest_result = ttest_ind(group1, group2, equal_var=False)
print('t-test statistics:', ttest_result.statistic)
print('p-value:', ttest_result.pvalue)

t-test statistics: 22.514758657461467
p-value: 8.933021714439036e-112


**Statement**: The t-test result for this hypothesis yields a t-test statistic of 22.51 and a p-value of approximately $8.93 \times 10^{-112}$. Since this p-value is extremely small and far below the conventional significance threshold of 0.05, there is extremely strong evidence to reject the null hypothesis. Therefore, the mean depth of earthquakes is significantly different between `mb` and `mw` magnitude type events in your dataset.

#### 3. Magnitude Consistency Across Months
Hypothesis: There is no significant association between the month of occurrence and the categorical bins of earthquake magnitude (such as low, medium, high).

In [ ]:
# Bin magnitudes into categories
df['mag_bin'] = pd.cut(df['mag'], bins=[0, 3.5, 5.0, 6.5, 9], labels=['Low', 'Medium', 'High', 'Very High'])

# Contingency table: mag_bin vs Month
contingency = pd.crosstab(df['mag_bin'], df['Month'])
chi2, p, dof, expected = chi2_contingency(contingency)
print('Chi-square statistic:', chi2)
print('Degrees of freedom:', dof)
print('p-value:', p)

Chi-square statistic: 46.542726462177434
Degrees of freedom: 33
p-value: 0.05922370884750974


**Statement**: The chi-square test result for this hypothesis yields a chi-square statistic of 46.54, degrees of freedom of 33, and a p-value of approximately 0.0592. Since this p-value is slightly above the conventional significance threshold of 0.05, there is insufficient evidence to reject the null hypothesis at the 5% level. Therefore, the data does not provide strong enough evidence to conclude that earthquake magnitude category and month of occurrence are dependent variables.

#### 4. DayOfWeek Consistency Across Event Type
Hypothesis: The type of event ('type' column: earthquake, quarry blast, etc.) is independent of the day of week
(DayOfWeek) on which it occurred.

In [ ]:
contingency = pd.crosstab(df['type'], df['DayOfWeek'])
chi2, p, dof, expected = chi2_contingency(contingency)
print('Chi-square statistic:', chi2)
print('Degrees of freedom:', dof)
print('p-value:', p)

Chi-square statistic: 92.16992465854213
Degrees of freedom: 36
p-value: 8.07286408381138e-07


**Statement**: The chi-square test result for this hypothesis yields a chi-square statistic of 92.17, degrees of freedom of 36, and a p-value of approximately $8.072 \times 10^{-07}$
 . Since this p-value is far below the conventional significance threshold of 0.05, there is very strong evidence to reject the null hypothesis. Therefore, the day of the week and event type are significantly dependent, indicating a meaningful association between these categorical variables in your data.

#### 5. Magnitude Mean Difference by Year Decade
Hypothesis: The mean earthquake magnitude (mag) differs significantly between records from two different decades (e.g., 2000s vs 2010s).

In [ ]:
df['decade'] = (df['Year'] // 10) * 10
dec1 = df[df['decade'] == 2000]['mag'].dropna()
dec2 = df[df['decade'] == 2010]['mag'].dropna()

ttest_result = ttest_ind(dec1, dec2, equal_var=False)
print('t-test statistic:', ttest_result.statistic)
print('p-value:', ttest_result.pvalue)

t-test statistic: 7.034358911416218
p-value: 2.0392846020245177e-12


**Statement**: The t-test result for this hypothesis yields a t-test statistic of 7.03 and a p-value of approximately $2.039 \times 10^{-12}$. Since this p-value is extremely small and far below the conventional significance threshold of 0.05, there is very strong evidence to reject the null hypothesis. Therefore, the mean earthquake magnitude differs significantly between the two compared decades in your dataset.

### Task:
- Suggest five additional hypothesis tests, based on the preprocessed earthquake dataset, that cover both numerical and categorical variables using t-tests and chi-square tests, to extend the depth of your analysis.

###1. Magnitude Difference by Day of Week (Weekday vs Weekend)
Hypothesis

H₀: No significant difference in mean earthquake magnitude between weekdays and weekends.

H₁: Significant difference in mean earthquake magnitude between weekdays and weekends.

In [ ]:
# Create weekend indicator (0 = weekday, 1 = weekend)
df['is_weekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

weekend_mag = df[df['is_weekend'] == 1]['mag']
weekday_mag = df[df['is_weekend'] == 0]['mag']

t_stat, p_val = stats.ttest_ind(weekend_mag, weekday_mag, nan_policy='omit')
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4f}")

if p_val < 0.05:
    print("Reject H₀ → Significant difference in magnitude between weekdays and weekends.")
else:
    print("Fail to reject H₀ → No significant difference in magnitude.")

T-statistic: 0.9967, P-value: 0.3189
Fail to reject H₀ → No significant difference in magnitude.


###2. RMS Variation Across Magnitude Types (t-test / ANOVA)

Goal: To test whether the average RMS (root mean square) value differs among different magnitude types.

H₀: The mean RMS values are the same across all magType categories.

H₁: At least one magType category has a different mean RMS value. (Numerical: rms, Categorical: magType)

In [ ]:
df_anova = df.dropna(subset=['rms', 'magType'])
groups = [group['rms'].values for name, group in df_anova.groupby('magType') if len(group) > 1]

if len(groups) > 1:
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"F-statistic: {f_stat:.4f}, P-value: {p_val:.4f}")
    if p_val < 0.05:
        print("Reject H₀ → RMS differs significantly among magnitude types.")
    else:
        print("Fail to reject H₀ → No significant RMS difference among magTypes.")
else:
    print("Not enough groups for ANOVA test.")

F-statistic: 335.1638, P-value: 0.0000
Reject H₀ → RMS differs significantly among magnitude types.


###3. Relationship Between Earthquake Status and Type (Chi-Square Test)

Goal: To check if the type of earthquake (e.g., "earthquake", "explosion") is related to how it was recorded (status: reviewed/automatic).

H₀: Earthquake type and status are independent.

H₁: Earthquake type and status are dependent (associated).

In [ ]:
cont_table = pd.crosstab(df['type'], df['status'])
chi2, p_val, dof, exp = chi2_contingency(cont_table)

print(f"Chi2: {chi2:.4f}, P-value: {p_val:.4f}")
if p_val < 0.05:
    print("Reject H₀ → Type and status are dependent (associated).")
else:
    print("Fail to reject H₀ → Type and status are independent.")

Chi2: 0.4683, P-value: 0.9982
Fail to reject H₀ → Type and status are independent.


###4. Average Depth Variation by Month (t-test / ANOVA)

Goal: To check if the average earthquake depth changes significantly across months.

H₀: The average depth is the same across all months.

H₁: There is a significant difference in depth between at least two months.

In [ ]:
df_anova_month = df.dropna(subset=['depth', 'Month'])
groups = [group['depth'].values for name, group in df_anova_month.groupby('Month') if len(group) > 1]

if len(groups) > 1:
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"F-statistic: {f_stat:.4f}, P-value: {p_val:.4f}")
    if p_val < 0.05:
        print("Reject H₀ → Depth varies significantly by month.")
    else:
        print("Fail to reject H₀ → No significant difference in average depth across months.")
else:
    print("Not enough groups for ANOVA test.")

F-statistic: 4.7521, P-value: 0.0000
Reject H₀ → Depth varies significantly by month.


###5. Magnitude Difference Between Northern and Southern Hemisphere (t-test)

Goal: To see if earthquakes in the Northern Hemisphere differ in magnitude from those in the Southern Hemisphere.

H₀: The average magnitude of earthquakes is the same in both hemispheres.

H₁: The average magnitude differs significantly between hemispheres.

In [ ]:
df['hemisphere'] = df['latitude'].apply(lambda x: 'Northern' if x >= 0 else 'Southern')

north_mag = df[df['hemisphere'] == 'Northern']['mag']
south_mag = df[df['hemisphere'] == 'Southern']['mag']

t_stat, p_val = stats.ttest_ind(north_mag, south_mag, nan_policy='omit')
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4f}")

if p_val < 0.05:
    print("Reject H₀ → Magnitude differs significantly between hemispheres.")
else:
    print("Fail to reject H₀ → No significant difference between hemispheres.")

T-statistic: 11.8584, P-value: 0.0000
Reject H₀ → Magnitude differs significantly between hemispheres.
